# A Safety Monitor for a Robot Arm Under Sensor Noise

This notebook is a guided tour of the study in this repository. We explain the setup in plain terms and reproduce the headline numbers from artifacts that already live in `results/` — no GPU or simulator needed.

**The question this study asks:** can we tell, *before* a robot tries to pick up an object, whether its camera is too degraded for the pick to succeed — and can we predict how that monitor will behave on kinds of noise it has never seen?

**Roadmap:**
1. The robot, the task, and what "success" means.
2. What can go wrong — and why predicting failure from raw appearance is hard.
3. The nine corruption types we study.
4. The safety monitor: a small CNN that watches the same image the policy sees and predicts the probability of failure.
5. The monitor in action — in-distribution predictions ranking severity correctly.
6. Leave-one-out: testing the monitor on corruption types it never saw.
7. Predicting *which* unseen corruptions the monitor will generalize to.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

REPO = Path('..').resolve()
RESULTS = REPO / 'results'
FIGURES = REPO / 'paper' / 'figures'
DOC_FIG = REPO / 'docs' / 'figures'
DEMO = RESULTS / 'demo_episodes'
print(f'Repo root: {REPO}')

## 1. The robot arm and the task

We use the **Google Robot** arm inside the SimplerEnv / ManiSkill simulator. The policy controlling the arm is **InternVLA-M1**, a vision-language-action model: it takes the wrist-camera image plus a natural-language instruction ("pick coke can") and outputs joint-space actions.

The task is `google_robot_pick_coke_can`: a coke can is placed at a *random* position on the table, and the policy has up to 100 steps to grasp it and lift it clear of the surface (success is determined by SimplerEnv's standard pick criterion — object lifted above the table while still grasped). Under clean conditions the policy succeeds **100 % of the time (50/50)**.

In [ ]:
# Canonical clean rollout — what the policy sees with no corruption.
display(Image(filename=str(DEMO / 'clean.gif')))

## 2. What goes wrong — and why predicting failure is hard

In the real world a robot's camera is almost never as clean as the simulator's. So we apply a **corruption** to every frame the policy sees before it sees it. The robot, the world, and the policy weights are unchanged — only the *appearance* of the image changes. This isolates the question: how robust is the policy to image-side perturbations alone?

And here is the central difficulty: **a heavily degraded image does not always mean the pick will fail, and a mildly degraded one does not always mean it will succeed**. Two examples at the *same* severity (budget 0.7) make the point.

### Limiting case — heavy rain (usually fails)

Rain droplets cover the lens, the coke can is partially occluded, and the policy loses track of it during the approach. Across 10 episodes the pick succeeds only **3 times**.

In [ ]:
display(Image(filename=str(DEMO / 'rain.gif')))

### Friendly case — dust on the lens (still succeeds)

Dust is visibly degrading the image too, but it diffuses light rather than blocking specific regions — the coke can's silhouette is preserved well enough that the policy still locates it. Across 10 episodes at the same budget 0.7, the pick succeeds **10 / 10 times**.

In [ ]:
display(Image(filename=str(DEMO / 'dust_camera.gif')))

### The full spread

Across all nine corruption types at budget 0.7, the success rate ranges from 30 % (rain) to 100 % (several types). The amount of visible degradation is a poor proxy for the policy's actual chance of failing — which motivates needing a learned monitor:

In [ ]:
summary = json.loads((DEMO / 'summary.json').read_text())
rows = sorted(summary.items(), key=lambda kv: kv[1]['success_rate'])
print(f"{'Corruption':<18}  {'Budget':>6}  {'Success rate (10 eps)':>22}")
print('-' * 50)
for t, r in rows:
    b = '   --' if r['budget'] is None else f"{r['budget']:>6.2f}"
    print(f"{t:<18}  {b}  {r['success_rate']:>22.0%}")

## 3. The nine corruption types

Our taxonomy spans seven physical mechanisms — lens contact, optical artifacts, blur, sensor noise, environmental contamination, illumination, and digital pipeline. Each row of the grid below shows the same scene under each corruption at increasing **budget** (severity from 0.1 to 0.9):

In [ ]:
fs = json.loads((RESULTS / 'loo_analysis_v2' / 'feature_similarity.json').read_text())
cats = fs['categories']
print(f"{'Category':<14}  {'Corruption':<16}")
print('-' * 32)
for t, c in sorted(cats.items(), key=lambda kv: (kv[1], kv[0])):
    print(f'{c:<14}  {t:<16}')

In [ ]:
display(Image(filename=str(FIGURES / 'corruption_grid.png')))

## 4. The safety monitor: a small CNN with a frozen backbone

The monitor is intentionally tiny: a **frozen ResNet-18** pre-trained on ImageNet (11 M params, never updated) followed by a trainable head — `512 → FC(64) → ReLU → Dropout(0.3) → FC(1) → sigmoid` — only **33 K trainable parameters** in total.

It looks at the same frame the policy looks at, and outputs a single number: the probability the pick about to be attempted will fail. It does this *before* the policy has acted.

In [ ]:
display(Image(filename=str(DOC_FIG / 'architecture.png')))

**Why the frozen backbone?** The hypothesis is that ImageNet features encode generic visual degradation cues — edges, textures, contrast, sharpness — that are perturbed in characteristic ways by any physical corruption mechanism. Only the 33 K-parameter head learns *which* perturbation patterns correlate with task failure. The monitor is therefore cheap to train, cheap to deploy, and — as we will see in §7 — its frozen feature space is exactly what lets us predict OOD generalization.

## 5. In-distribution predictions — the easy case

We first ask the easy question: if the monitor has *seen* each corruption type during training, does its predicted P(failure) actually rank severity? This is the in-distribution check.

The summary figure plots mean predicted failure probability versus budget for every corruption type. The bars rise monotonically with severity — at low budgets the monitor predicts low failure, at high budgets it predicts high failure — as we want:

In [ ]:
display(Image(filename=str(DOC_FIG / 'evaluation_summary.png')))

Frame-by-frame, the monitor's prediction tracks the rollout. Below is a heavy-rain episode with the monitor's P(failure) gauge overlaid in the corner — the gauge climbs as the rain occludes the can and the policy loses tracking:

In [ ]:
display(Image(filename=str(DOC_FIG / 'demo_rain.gif')))

And for a forgiving case where the policy still succeeds — motion blur. The monitor predicts low P(failure) throughout, the pick goes through:

In [ ]:
display(Image(filename=str(DOC_FIG / 'demo_motion_blur.gif')))

## 6. Leave-one-out: out-of-distribution generalization

In-distribution is the easy case — by construction. The deployment-relevant question is: **what happens when a corruption type the monitor has never seen appears?** We test this with **leave-one-out (LOO)**: for each of the nine corruption types, train the monitor on episodes from the other eight, then test on the held-out one.

We score each held-out fold with:
* **Spearman ρ** — how well the monitor *ranks* episodes (high-risk vs. low-risk).
* **AUROC** — how well it *separates* eventual failures from successes.

In [ ]:
loo = json.loads((RESULTS / 'loo_analysis_v3' / 'loo_summary.json').read_text())
print(f"{'Held out':<15}  {'Spearman':>9}  {'AUROC':>6}")
print('-' * 35)
for r in loo['per_fold']:
    rho = r['spearman_rho']
    auc = r['auroc']
    rho_s = '   n/a' if rho is None or rho != rho else f'{rho:+.2f}'
    auc_s = '   n/a' if auc is None or auc != auc else f'{auc:.2f}'
    print(f"{r['held_out']:<15}  {rho_s:>9}  {auc_s:>6}")
print('-' * 35)
print(f"{'mean':<15}  {loo['mean_rho']:+.2f}      {loo['mean_auroc']:.2f}")

Three of the nine folds are degenerate — the policy either failed every episode or succeeded every episode under that corruption at the budgets we tested, so neither ρ nor AUROC is defined. Of the six folds with signal, the monitor's mean AUROC is **0.92**: usable on unseen noise, but with real variance — `fingerprint` is the hard case (AUROC 0.64) while `motion_blur` and `low_light` are essentially perfect (1.0).

Sanity check on one fold: even though `rain` was *held out* of training, the monitor's predictions still separate failures from successes:

In [ ]:
fold = json.loads((RESULTS / 'loo_analysis_v3' / 'fold_rain.json').read_text())
preds, labels = [], []
for budget, br in fold['eval_results'].items():
    for ep in br['episodes']:
        preds.append(ep['predicted_p_failure_mean'])
        labels.append(ep['actual_failure'])
preds, labels = np.array(preds), np.array(labels)
print(f'Held-out corruption: rain  ({len(preds)} episodes)')
print(f'Mean predicted P(failure) on episodes that succeeded: {preds[labels==0].mean():.2f}')
print(f'Mean predicted P(failure) on episodes that failed:    {preds[labels==1].mean():.2f}')

Which raises the obvious question: **could we have predicted that ranking ahead of time?** Why does `motion_blur` work perfectly while `fingerprint` is hard?

## 7. Predicting OOD performance from feature-space geometry

Here is the central new finding. For each pair of corruption types, we computed two cosine similarities in the ResNet-18 feature space:

* **raw similarity** — between the corrupted frames themselves.
* **delta similarity** — between the *differences* $g(\text{corrupted}) - g(\text{clean})$. This isolates *what each corruption changed about the image*, dropping the parts they share with the clean baseline.

Then for each held-out corruption we asked: what is the maximum delta-similarity to any corruption that *was* in the training set? Intuitively, if the held-out fingerprint smudge looks (in feature-delta space) a lot like some training corruption, the monitor should generalize to it. If it is geometrically novel, it should not.

In [ ]:
display(Image(filename=str(RESULTS / 'loo_analysis_v3' / 'ood_predictability.png')))

In [ ]:
ood = json.loads((RESULTS / 'loo_analysis_v3' / 'ood_predictability.json').read_text())
c = ood['correlations']
print(f"n = {ood['n_valid']} folds with signal\n")
print(f"{'predictor':<25}  {'vs.':<6}  {'Pearson':>8}  {'Spearman':>9}")
print('-' * 56)
for k, label in [
    ('max_diff_vs_auroc',  'AUROC'),
    ('mean_diff_vs_auroc', 'AUROC'),
    ('max_raw_vs_auroc',   'AUROC'),
    ('max_diff_vs_rho',    'rho'),
    ('max_raw_vs_rho',     'rho'),
]:
    name = k.replace('_vs_' + label.lower(), '').replace('_', ' ')
    print(f"{name:<25}  {label:<6}  {c[k]['pearson']:+8.2f}  {c[k]['spearman']:+9.2f}")

Two things to notice:

1. **Delta-similarity to the training set predicts held-out AUROC** at Pearson +0.85, Spearman +0.82. A corruption whose feature-delta is close to something the monitor has already seen is one the monitor will handle.
2. **Raw similarity is actively misleading.** Two corruptions can have nearly identical raw features (both are mostly the underlying scene) while their *deltas* point in totally different directions. The Spearman against ρ is −0.93 — high raw similarity goes with *worse* OOD ranking. The lesson is that what matters is *what the corruption changed*, not what the image looks like overall.

Operationally, this gives us a free pre-flight check: before deploying the monitor against a brand-new corruption, take a handful of clean/corrupted frame pairs, compute the feature-delta, compare it to the training-set deltas. If the maximum cosine similarity is comfortably above the training distribution, expect a usable monitor. If not, collect labels before trusting it.

## Where to go next

* `paper/main.tex` — full write-up with calibration analysis (ECE = 0.243 pooled), the per-fold reliability diagrams, and discussion.
* `adversarial_dust/safety_predictor.py` — the CNN definition and training loop.
* `adversarial_dust/run_safety_predictor.py` — the orchestration script (`--loo --skip-collect --run-baselines`).
* `scripts/compute_ece.py` — calibration analysis used in §VII of the paper.
* `scripts/predict_ood_from_similarity.py` — exactly the analysis in §7 above.
* `results/loo_analysis_v3/` — every artifact this notebook reads.